# V7_D_N04 — Shelter Before Disaster

**SRAI Book 7 — Applied AI Studio: National Sector Intelligence**  
Controlled draft using synthetic data. Outputs support authorized human review; they do not constitute official declarations or automated decisions.

## Decision contract
Support preparedness, field verification, accessible shelter planning, and proportional response. Owners: disaster-management and municipal authorities. The notebook does not order evacuation or determine entitlement.

In [1]:
import numpy as np,pandas as pd
rng=np.random.default_rng(7404); n=90; zones=pd.DataFrame({'zone':[f'Z{i:03d}' for i in range(n)],'population':rng.integers(300,5200,n),'hazard_prob':rng.uniform(.03,.55,n),'vulnerability':rng.uniform(.15,.95,n),'warning_access':rng.uniform(.35,1,n),'mobility_constraint_share':rng.uniform(.03,.28,n),'data_age_months':rng.integers(1,25,n)}); shelters=pd.DataFrame({'shelter':[f'H{i:02d}' for i in range(14)],'capacity':rng.integers(250,1200,14),'accessibility':rng.uniform(.55,1,14)}); zones.head().round(2)

## Evidence contract
Hazard, exposure, and vulnerability are distinct. Population grids, censuses, building data, shelters, accessibility, and routes require current authoritative sources and field validation.

In [2]:
assert zones.zone.is_unique and shelters.shelter.is_unique; zones['quality_pass']=zones.data_age_months<=18; print('CURRENT_SHARE',round(zones.quality_pass.mean(),2),'TOTAL_CAPACITY',shelters.capacity.sum())

CURRENT_SHARE 0.74 TOTAL_CAPACITY 9362


## Expected affected population is a scenario metric
Multiplying hazard probability, population, and vulnerability supports comparison under assumptions; it is not a certain casualty or displacement forecast.

In [3]:
zones['expected_affected']=zones.population*zones.hazard_prob*zones.vulnerability; zones['priority']=zones.expected_affected*(1+.5*(1-zones.warning_access)+.5*zones.mobility_constraint_share); zones.nlargest(8,'priority')[['zone','population','hazard_prob','vulnerability','expected_affected','priority']].round(1)

## Quality abstention and verification list
Stale zones are not treated as current truth. High-priority current zones receive preparedness review; stale high-priority zones receive urgent verification.

In [4]:
zones['disposition']=np.where(~zones.quality_pass,'VERIFY—STALE EVIDENCE',np.where(zones.priority>=zones[zones.quality_pass].priority.quantile(.8),'PREPAREDNESS REVIEW','MONITOR')); print(zones.disposition.value_counts().to_string())

disposition
MONITOR                  53
VERIFY—STALE EVIDENCE    23
PREPAREDNESS REVIEW      14


## Accessible shelter capacity
Nominal capacity overstates usable capacity when accessibility, staffing, water, sanitation, safety, or routes are inadequate.

In [5]:
shelters['effective_capacity']=(shelters.capacity*shelters.accessibility).astype(int); demand=zones.loc[zones.quality_pass,'expected_affected'].sum(); nominal=shelters.capacity.sum(); effective=shelters.effective_capacity.sum(); print({'scenario_demand':round(demand),'nominal_capacity':nominal,'effective_capacity':effective,'effective_gap':max(0,round(demand-effective))})

{'scenario_demand': 23907, 'nominal_capacity': np.int64(9362), 'effective_capacity': np.int64(6856), 'effective_gap': 17051}


## Scenario uncertainty
Compare lower/reference/upper hazard multipliers. This is more informative than presenting one precise requirement.

In [6]:
scenario=[]
for label,m in [('lower',.7),('reference',1),('upper',1.35)]:
 d=(zones.loc[zones.quality_pass,'population']*(zones.loc[zones.quality_pass,'hazard_prob']*m).clip(upper=1)*zones.loc[zones.quality_pass,'vulnerability']).sum(); scenario.append({'scenario':label,'affected':round(d),'capacity_gap':max(0,round(d-effective))})
print(pd.DataFrame(scenario).to_string(index=False))

 scenario  affected  capacity_gap
    lower     16735          9879
reference     23907         17051
    upper     32274         25418


## Rights and operational safeguards
Plans must include accessible communication, transport, family unity, disability inclusion, protection, grievance routes, data minimization, and no discriminatory exclusion.

In [7]:
plan={'status':'PREPAREDNESS SCENARIO—AUTHORITY REVIEW REQUIRED','field_verification':zones[zones.disposition!='MONITOR'].zone.tolist()[:12],'safeguards':['accessible alerts','assisted transport','family unity','protection and privacy','non-discrimination','grievance route'],'prohibited_use':'automatic evacuation, detention, exclusion, or benefit denial'}; print(plan)

{'status': 'PREPAREDNESS SCENARIO—AUTHORITY REVIEW REQUIRED', 'field_verification': ['Z004', 'Z007', 'Z010', 'Z011', 'Z013', 'Z019', 'Z022', 'Z023', 'Z025', 'Z026', 'Z028', 'Z029'], 'safeguards': ['accessible alerts', 'assisted transport', 'family unity', 'protection and privacy', 'non-discrimination', 'grievance route'], 'prohibited_use': 'automatic evacuation, detention, exclusion, or benefit denial'}


## Exercises
1. Add route/travel-time constraints. 2. Model shelter failure. 3. Add uncertainty in population estimates. 4. Explain why the highest exposure zone may not receive the first intervention.

## Exact solutions
1. Use authoritative networks, hazards, modes, barriers, congestion, and accessibility. 2. Stress-test closure/capacity loss and identify redundancy. 3. propagate census/grid uncertainty through scenario totals and rank stability. 4. Urgency also depends on warning time, feasibility, vulnerability, access, cascading hazards, current evidence, and authorized response strategy.

In [8]:
assert len(plan['safeguards'])>=5 and len(scenario)==3; assert 'automatic evacuation' in plan['prohibited_use']; print('V7_D_N04_COMPLETE_EXECUTION_PASS')

V7_D_N04_COMPLETE_EXECUTION_PASS
